## U30s from Harris wind spectrum

Each HL folder contains 9 conditions, each with a different U10.
The wind time series is identical across HL folders for the same condition index,
so a single HL folder is sufficient to extract all 9 (U10, U30s) pairs.

**Method:** For each condition, split `Velocityinmaindirection` into 30-second windows
and take the highest window mean. When multiple seeds are available, the max is taken
across all seeds.

In [ ]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

# Resolve this notebook's folder (works in VS Code and standard Jupyter)
BASE    = Path(globals().get('__vsc_ipynb_file__', Path.cwd() / '_')).parent
HEADING = "0deg"
REF_HL  = "HL7"          # any HL folder works — wind series is the same in all
REF_RUN = 1              # SIMA run number for REF_HL

DT       = 0.5           # seconds per sample
WINDOW_S = 30            # averaging window [s]
SPW      = int(WINDOW_S / DT)  # samples per window = 60

In [2]:
hl_dir     = BASE / HEADING / REF_HL
seed_files = sorted(hl_dir.glob(f"DynamicResults_{REF_HL}_run{REF_RUN}-*.h5"))
print(f"Seeds found: {[f.name for f in seed_files]}")

# Collect wind series per condition across all seeds
# cond_series[i] = list of arrays (one per seed)
cond_series = {}

for fpath in seed_files:
    with h5py.File(fpath, 'r') as f:
        root       = list(f.keys())[0]
        run_group  = f[f"{root}/Run{REF_RUN}"]
        cond_keys  = sorted(run_group.keys())
        for cond_key in cond_keys:
            arr = f[
                f"{root}/Run{REF_RUN}/{cond_key}"
                "/Dynamic/Tanker/Wind velocity/Velocityinmaindirection"
            ][:]
            cond_series.setdefault(cond_key, []).append(arr)

# Build results table
records = []
for cond_key in sorted(cond_series):
    max_u30s = 0.0
    u10_values = []
    for arr in cond_series[cond_key]:
        u10_values.append(arr.mean())
        n_win   = len(arr) // SPW
        windows = arr[: n_win * SPW].reshape(n_win, SPW)
        max_u30s = max(max_u30s, windows.mean(axis=1).max())
    records.append({
        "Condition": cond_key,
        "U10 [m/s]": round(np.mean(u10_values), 4),
        "U30s [m/s]": round(max_u30s, 4),
    })

df = pd.DataFrame(records)
df

Seeds found: ['DynamicResults_HL7_run1-1.h5', 'DynamicResults_HL7_run1-2.h5', 'DynamicResults_HL7_run1-3.h5', 'DynamicResults_HL7_run1-4.h5', 'DynamicResults_HL7_run1-5.h5']


,Condition,U10 [m/s],U30s [m/s]
0,Run1_1,4.5017,5.8963
1,Run1_2,5.8032,7.6012
2,Run1_3,6.8844,8.9940
3,Run1_4,7.7852,10.1389
4,Run1_5,8.6060,11.1704
5,Run1_6,9.3667,12.1173
6,Run1_7,10.0368,12.2123
7,Run1_8,10.6767,12.9656
8,Run1_9,11.3066,13.7035
